# Pipeline

Main pipeline notebook. All logic lives in `pipeline/`; this notebook handles configuration and orchestration only.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import pandas as pd
import os
import sys

from pathlib import Path

# Moving up to the project root to ensure imports work correctly regardless of execution context
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from pathlib import Path
from openai import OpenAI
from utilities import (
    OPENAI_API_KEY,
    TASK_STATEMENTS_PATH,
    MAJOR_CATEGORIES,
    WORK_RELATED_OUTPUT_PATH,
    TIMEZONES_OUTPUT_PATH,
    TASK_MAPPING_OUTPUT_PATH,
    LABOR_TRANSFER_OUTPUT_PATH,
    JOB_ZONES_PATH,
    FINAL_OUTPUT_PATH,
    ExecutionMode,
)
from pipeline import (
    load_wildchat,
    sample_conversations,
    preprocess_conversations,
    filter_work_conversations,
    find_timezones,
    normalize_timezone,
    map_conversation_to_task,
    filter_task_mappings,
    analyze_labor_transfer,
    expand_labor_transfer_labels,
)

In [2]:
# Device setup
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [3]:
# API client setup
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
execution_mode = ExecutionMode.BATCH

## 2. Data Loading

In [4]:
english_conversations = load_wildchat()
total_rows = len(english_conversations)
print(f"Total English conversations: {total_rows}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total English conversations: 1679371


In [5]:
sample_df = sample_conversations(english_conversations)
print(f"Sample shape: {sample_df.shape}")

Sample shape: (58777, 14)


In [6]:
sample_conversations_df = preprocess_conversations(sample_df)
print(f"After dedup: {sample_conversations_df.shape}")
sample_conversations_df.head(2)

After dedup: (52489, 5)


,conversation,timestamp,country,state,hashed_ip
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...


## 3. Work-Related Conversation Filtering

In [ ]:
if not WORK_RELATED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        client=client,
        conversations=sample_conversations_df,
        path=WORK_RELATED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
    answers_df = pd.DataFrame(
        {
            "conversation": sample_conversations_df["conversation"],
            "is_work_related_model": answers,
        }
    )
    answers_df.to_csv(WORK_RELATED_OUTPUT_PATH, index=False)
else:
    answers_df = pd.read_csv(WORK_RELATED_OUTPUT_PATH)

sample_conversations_df = sample_conversations_df.merge(
    answers_df, on="conversation", how="left"
)

work_related_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()

print(f"Work-related conversations: {work_related_df.shape}")
work_related_df.head(2)

## 4. Timezone conversion

In [ ]:
if not TIMEZONES_OUTPUT_PATH.exists():
    work_related_df = find_timezones(df=work_related_df)
    work_related_df.to_csv(TIMEZONES_OUTPUT_PATH, index=False)
else:
    work_related_df = pd.read_csv(TIMEZONES_OUTPUT_PATH)

work_related_df = work_related_df[work_related_df["timezone"].notnull()]
print(f"After timezone filter: {work_related_df.shape}")

In [ ]:
work_related_df = normalize_timezone(df=work_related_df)
work_related_df[["timestamp", "timezone", "timestamp_local"]].head(3)

## 5. Task Mapping

In [ ]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)
tasks_df.head()

In [ ]:
if not TASK_MAPPING_OUTPUT_PATH.exists():
    task_mapped_df = map_conversation_to_task(
        client=client,
        conversations=work_related_df,
        tasks=tasks_df,
        path=TASK_MAPPING_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
else:
    task_mapped_df = pd.read_csv(TASK_MAPPING_OUTPUT_PATH)

print(f"Task mapped conversations: {task_mapped_df.shape}")
task_mapped_df.head(2)

In [ ]:
task_mapped_df = filter_task_mappings(df=task_mapped_df, column_name="tasks")

task_mapped_df["job_title"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[0] if pd.notnull(x) else None
)
task_mapped_df["selected_task"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[1] if pd.notnull(x) else None
)

print(f"After consensus filter: {task_mapped_df.shape}")
task_mapped_df.head(2)

In [ ]:
task_mapped_df["conversation_str"] = task_mapped_df["conversation"].astype(str)
work_related_df["conversation_str"] = work_related_df["conversation"].astype(str)

task_mapped_df = task_mapped_df.merge(
    work_related_df.drop(columns=["conversation"]), on="conversation_str", how="inner"
)

task_mapped_df = task_mapped_df.drop(columns=["conversation_str"])

print(f"Final task mapped DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

## 6. Labor Transfer Analysis

In [ ]:
if not LABOR_TRANSFER_OUTPUT_PATH.exists():
    labor_transfer_labels = analyze_labor_transfer(
        client=client, df=task_mapped_df, execution_mode=execution_mode
    )
    pd.DataFrame({"label": labor_transfer_labels}).to_csv(
        LABOR_TRANSFER_OUTPUT_PATH, index=False
    )
else:
    labor_transfer_labels = pd.read_csv(LABOR_TRANSFER_OUTPUT_PATH)["label"].tolist()

print(f"Labor transfer labels: {len(labor_transfer_labels)}")
task_mapped_df["labor_transfer"] = labor_transfer_labels
print(f"Labor transfer labels assigned: {task_mapped_df.shape}")

In [ ]:
task_mapped_df = expand_labor_transfer_labels(
    df=task_mapped_df, label_column="labor_transfer"
)
print(f"Final DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

# 7. Job Zones

In [ ]:
job_zones_df = pd.read_excel(JOB_ZONES_PATH)
job_zones_df.head(2)

In [ ]:
task_mapped_df = task_mapped_df.merge(
    job_zones_df[["Title", "Job Zone"]],
    left_on="job_title",
    right_on="Title",
    how="left",
)
task_mapped_df = task_mapped_df.drop(columns=["Title"])
print(f"After merging job zones: {task_mapped_df.shape}")
task_mapped_df.head(2)

In [ ]:
final_df = task_mapped_df.copy()
final_df.to_csv(FINAL_OUTPUT_PATH, index=False)